# Knowledge 6 - API do IBGE

A API do IBGE é meio lixo supremo, mas da pra tirar algumas coisas interessantes.


### Conceitos importantes de hoje:

 * APIs com endpoints diferentes
 * Tratamento de dados

### 2 informações importantes obtidas pela API:
 
 * Projeção populacional por país ou regiões
 * Indicadores econômicos por país
 
### Documentação geral

https://servicodados.ibge.gov.br/api/docs


# Projeção populacional por país ou regiões

### Documentação:

https://servicodados.ibge.gov.br/api/docs/projecoes

## URL Base

https://servicodados.ibge.gov.br/api/v1/projecoes/populacao/{localidade}

Localidade deve ser:

    * País: BR
    * Regiões: 
        ** 1 - Norte
        ** 2 - Nordeste
        ** 3 - Sudeste
        ** 4 - Sul
        ** 5 - Centro-Oeste

In [ ]:
%pip install pandas

In [ ]:
%pip install requests

In [2]:
import requests
import pandas as pd
import json

In [16]:
paises = 'BR'
indicadores= 77819|77820
url = f'https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}'

response = requests.get(url)
data = json.loads(response.text)
print(data)

[{'id': 77823, 'indicador': 'Economia - PIB per capita', 'unidade': {'id': 'US$', 'classe': 'N', 'multiplicador': 1}, 'series': [{'pais': {'id': 'BR', 'nome': 'Brasil'}, 'serie': [{'-': None}, {'1990': '3117.74'}, {'1990-1995': None}, {'1995': '4756.75'}, {'1995-2000': None}, {'1999-2001': None}, {'2000': '3766.55'}, {'2000-2002': None}, {'2000-2005': None}, {'2001': '3176.29'}, {'2001-2003': None}, {'2002': '2855.94'}, {'2002-2004': None}, {'2003': '3090.61'}, {'2003-2005': None}, {'2004': '3663.82'}, {'2004-2006': None}, {'2005': '4827.78'}, {'2005-2007': None}, {'2005-2010': None}, {'2006': '5934.14'}, {'2006-2008': None}, {'2007': '7409.69'}, {'2007-2009': None}, {'2008': '8908.33'}, {'2008-2010': None}, {'2009': '8678.66'}, {'2009-2011': None}, {'2010': '11403.28'}, {'2010-2012': None}, {'2010-2015': None}, {'2011': '13396.62'}, {'2011-2013': None}, {'2012': '12521.72'}, {'2012-2014': None}, {'2013': '12458.89'}, {'2013-2015': None}, {'2014': '12274.99'}, {'2014-2016': None}, {'20

# Indicadores econômicos por país

### Documentação: 

https://servicodados.ibge.gov.br/api/docs/paises

## URL Base

https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}


obs: Quando desejar mais de um indicador, ou país, utilizar o "|"

Localidade deve ser:

    * País: Código de 2 digitos de acordo com  norma ISO 3166-1 ALPHA-2 (Exemplo: BR, US)

Indicador deve ser:

    * Um número de acordo com o site do IBGE (ele define id's)

obs: Na documentação tem o id dos indicadores e o código de cada país

In [20]:
paises = 'BR|US'
indicadores = '77823' #PIB per capita
nome_pais = ['Brasil', 'EUA']


url = f'https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}'

response = requests.get(url)
data = json.loads(response.text)

lista_dfs = []

for i, nome_pais in enumerate(nome_pais):
    
    lista_anos = []
    lista_valores = []

    for informacoes in data[0]['series'][i]['serie']: #no lugar desse "0", caso você puxe dois indicadores ao mesmo tempo, vai entrar um outro loop.

        valores = list(informacoes.items())
        lista_anos.append(valores[0][0])
        lista_valores.append(valores[0][1])

    df = pd.DataFrame(list(zip(lista_anos,lista_valores)), columns=["Anos",f"{nome_pais}"]).dropna()

    lista_dfs.append(df)
    
df_final = pd.merge(lista_dfs[0],lista_dfs[1], on='Anos')
df_final = df_final.set_index("Anos")

display(df_final)

,Brasil,EUA
Anos,,
1990,3117.74,23888.60
1995,4756.75,28690.88
2000,3766.55,36329.97
2001,3176.29,37133.62
2002,2855.94,37997.74
2003,3090.61,39490.30
2004,3663.82,41724.64
2005,4827.78,44123.40
2006,5934.14,46301.99


In [18]:
data

[{'id': 77823,
  'indicador': 'Economia - PIB per capita',
  'unidade': {'id': 'US$', 'classe': 'N', 'multiplicador': 1},
  'series': [{'pais': {'id': 'BR', 'nome': 'Brasil'},
    'serie': [{'-': None},
     {'1990': '3117.74'},
     {'1990-1995': None},
     {'1995': '4756.75'},
     {'1995-2000': None},
     {'1999-2001': None},
     {'2000': '3766.55'},
     {'2000-2002': None},
     {'2000-2005': None},
     {'2001': '3176.29'},
     {'2001-2003': None},
     {'2002': '2855.94'},
     {'2002-2004': None},
     {'2003': '3090.61'},
     {'2003-2005': None},
     {'2004': '3663.82'},
     {'2004-2006': None},
     {'2005': '4827.78'},
     {'2005-2007': None},
     {'2005-2010': None},
     {'2006': '5934.14'},
     {'2006-2008': None},
     {'2007': '7409.69'},
     {'2007-2009': None},
     {'2008': '8908.33'},
     {'2008-2010': None},
     {'2009': '8678.66'},
     {'2009-2011': None},
     {'2010': '11403.28'},
     {'2010-2012': None},
     {'2010-2015': None},
     {'2011': '1

In [22]:

# Converte as colunas para números decimais
df_final['EUA'] = df_final['EUA'].astype(float)
df_final['Brasil'] = df_final['Brasil'].astype(float)

# Agora a sua divisão vai funcionar
df_final['eua_maior'] = df_final['EUA'] / df_final['Brasil']
df_final

df_final

,Brasil,EUA,eua_maior
Anos,,,
1990,3117.74,23888.60,7.662153
1995,4756.75,28690.88,6.031614
2000,3766.55,36329.97,9.645424
2001,3176.29,37133.62,11.690878
2002,2855.94,37997.74,13.304810
2003,3090.61,39490.30,12.777510
2004,3663.82,41724.64,11.388289
2005,4827.78,44123.40,9.139480
2006,5934.14,46301.99,7.802645


# Exercícios 

* Exercício 139: Colete os dados de indivíduos com acesso a internet no BR na API de indicadores.
* Exercício 140: Colete os nomes mais registrados no Brasil desde 1950.

In [ ]:
#gabarito 139

paises = 'BR'
indicadores = '77857' #Pessoas com internet

url = f'https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}'

response = requests.get(url)
data = json.loads(response.text)

lista_dfs = []    
lista_anos = []
lista_valores = []

for informacoes in data[0]['series'][0]['serie']: 
        
        #no lugar desse "0", caso você puxe dois indicadores ao mesmo tempo, vai entrar um outro loop.

        valores = list(informacoes.items())
        lista_anos.append(valores[0][0])
        lista_valores.append(valores[0][1])

df = pd.DataFrame(list(zip(lista_anos,lista_valores)), columns=["Anos", "Brasil"]).dropna()
df = df.set_index("Anos")

print(df)

In [ ]:
#gabarito 140 

url = 'https://servicodados.ibge.gov.br/api/v2/censos/nomes/ranking'

response = requests.get(url)
data = json.loads(response.text)

lista_df = []

for informacoes in data[0]['res']: 

    df = pd.DataFrame(informacoes, index = [0])
    
    lista_df.append(df)
    
rankings_nomes = pd.concat(lista_df, ignore_index = True)

rankings_nomes